10: Fine-tune T5-small for Summarization

In [15]:
import torch
from datasets import load_dataset
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    Trainer, 
    TrainingArguments,
    DataCollatorForSeq2Seq
)

# 1. Load Real Dataset from Hugging Face (CNN/DailyMail News Summarization)
print("Loading CNN/DailyMail dataset from Hugging Face...")
raw_train_dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:100]")
raw_eval_dataset = load_dataset("cnn_dailymail", "3.0.0", split="validation[:10]")

# 2. Tokenizer & Pre-trained Model Loading
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# 3. Tokenization & Preprocessing Function
def preprocess_function(examples):
    # Prefix required for T5 model
    inputs = ["summarize: " + article for article in examples["article"]]
    
    # Tokenize input articles
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    
    # Tokenize target summaries (highlights)
    labels = tokenizer(text_target=examples["highlights"], max_length=128, truncation=True)
    
    # Replace padding token ids with -100 so PyTorch cross-entropy loss ignores padding
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] 
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels_ids
    return model_inputs

print("Preprocessing and tokenizing dataset...")
train_dataset = raw_train_dataset.map(preprocess_function, batched=True, remove_columns=raw_train_dataset.column_names)
eval_dataset = raw_eval_dataset.map(preprocess_function, batched=True, remove_columns=raw_eval_dataset.column_names)

# 4. Data Collator for Dynamically Padding Batches
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. Training Arguments Configuration
training_args = TrainingArguments(
    output_dir="./t5_cnn_summarization_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=1e-4,  # Safe learning rate to preserve pre-trained knowledge
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    use_cpu=not torch.cuda.is_available()  # Automatic GPU/CPU detection
)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# Execute Fine-Tuning
print("Starting T5-Small Fine-Tuning on CNN/DailyMail dataset...")
trainer.train()
print("Fine-tuning Complete!")

# 7. Test Fine-Tuned Model Inference
device = model.device

sample_article = (
    "summarize: Artificial intelligence and modern large language models have transformed processing speeds across "
    "various industries. Researchers have demonstrated that combining retrieval techniques with neural network "
    "generation significantly reduces model hallucination while expanding contextual understanding."
)

inputs = tokenizer(sample_article, return_tensors="pt", max_length=512, truncation=True).to(device)

# Generate Summary with Beam Search
outputs = model.generate(
    inputs.input_ids,
    max_length=60,
    min_length=10,
    num_beams=4,
    no_repeat_ngram_size=2,
    early_stopping=True
)

summary_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== GENERATED SUMMARY ===")
print(summary_output)

Loading CNN/DailyMail dataset from Hugging Face...


Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Preprocessing and tokenizing dataset...


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Starting T5-Small Fine-Tuning on CNN/DailyMail dataset...


Epoch,Training Loss,Validation Loss
1,2.228900,2.412191
2,2.077100,2.407578
3,2.489500,2.403599


Fine-tuning Complete!

=== GENERATED SUMMARY ===
combining retrieval techniques with neural network generation significantly reduces model hallucination while expanding contextual understanding.
